In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from pathlib import Path

from analysis.utils.utils import get_weights_path, get_figures_path

# ── Config ────────────────────────────────────────────────────────────────────
HISTORY_FILE = get_weights_path() / "eval_history.jsonl"

# Leave empty to compare ALL runs, or list weights_dir strings to filter:
# e.g. COMPARE = ["gravnet_regression_faser_all_events_mean_reparam_std",
#                 "gravnet_regression_faser_all_events_mean_reparam_std_huber1.0"]
COMPARE = [
    "gravnet_regression_faser_huber1.0_nofaser_final",           # Exp1 baseline
    "gravnet_regression_faser_huber1.0_truth2_nofaser_final",    # Exp4 truth-label ceiling
    "gravnet_regression_faser_huber1.0_binaryprob_nofaser_final",# Exp5 full pipeline
]

TARGETS      = ["E_nu", "E_lepton", "E_roe"]
TARGET_LATEX = [r"$E_\nu$", r"$E_\mathrm{lep}$", r"$E_\mathrm{roe}$"]
ENERGY_BINS_TEV = [
    (0.01, 0.05), (0.05, 0.1), (0.1, 0.2),
    (0.2,  0.3),  (0.3,  0.5), (0.5,  0.7),
    (0.7,  1.0),  (1.0,  1.5), (1.5,  3.0),
]

_COLORS = ["#4B9BEB", "#FC5C5C", "#3EC679"]
_ALPHA  = 0.75

sns.set_style("ticks")
sns.set_context("paper", font_scale=1.5)
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['DejaVu Serif', 'Times New Roman', 'Times'],
    'mathtext.fontset': 'dejavuserif',
    'axes.linewidth': 0.8,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'figure.dpi': 200,
})

In [ ]:
# ── Logic to handle auto-selection ──────────────────────────────────────────
if not COMPARE:
    try:
        all_entries = []
        with open(HISTORY_FILE, "r") as f:
            for line in f:
                if line.strip():
                    all_entries.append(json.loads(line))
        
        # Sort by timestamp (ISO format strings sort correctly)
        all_entries.sort(key=lambda x: x["timestamp"])
        
        # Get unique weights_dir values while preserving recent order
        unique_dirs = []
        for entry in reversed(all_entries):
            wd = entry["weights_dir"]
            if wd not in unique_dirs:
                unique_dirs.append(wd)
            if len(unique_dirs) == 2:
                break
        
        COMPARE = unique_dirs
        print(f"Auto-selected recent runs: {COMPARE}")
        
    except FileNotFoundError:
        print(f"Warning: {HISTORY_FILE} not found.")
    except Exception as e:
        print(f"Error parsing history: {e}")

In [ ]:
def short_name(weights_dir):
    """Strip common prefix for display."""
    prefix = "gravnet_regression_faser_"
    return weights_dir.replace(prefix, "") if weights_dir.startswith(prefix) else weights_dir

_FINAL_LABELS = {
    "huber1.0_nofaser_final":            "GravNet",
    "huber1.0_binaryprob_nofaser_final": "GravNet + classifier",
    "huber1.0_truth2_nofaser_final":     "GravNet + truth-labels",
}

def display_name(r):
    sn = short_name(r["weights_dir"])
    return _FINAL_LABELS.get(sn, sn)

# ── Load history ──────────────────────────────────────────────────────────────
all_runs = [json.loads(l) for l in open(HISTORY_FILE) if l.strip()]

# Deduplicate by cache_key (keep last occurrence)
seen = {}
for r in all_runs:
    seen[r["cache_key"]] = r
all_runs = list(seen.values())

if COMPARE:
    # Keep only the most recent entry per weights_dir, in the order of COMPARE
    by_wd = {}
    for r in all_runs:
        wd = r["weights_dir"]
        if wd in COMPARE:
            if wd not in by_wd or r["timestamp"] > by_wd[wd]["timestamp"]:
                by_wd[wd] = r
    runs = [by_wd[wd] for wd in COMPARE if wd in by_wd]
else:
    runs = all_runs

print(f"Loaded {len(all_runs)} unique entries from history, comparing {len(runs)}.")
print()
for i, r in enumerate(runs):
    c = r["checkpoint"]
    loss = c.get("loss_fn") or "mse"
    delta = f"δ={c['huber_delta']}" if loss == "huber" and c.get("huber_delta") else ""
    print(f"[{i}] {r['weights_dir']}")
    print(f"     loss={loss}{delta}  epoch={c['epoch']}  val_loss={c['val_loss']:.4f}  val_rmse={c['val_rmse']:.4f}  ts={r['timestamp']}")

import os
_names = [display_name(r) for r in runs]
_prefix = os.path.commonprefix(_names)
_col_names = [n[len(_prefix):].strip('_') or 'base' for n in _names]

_comparison_key = "_vs_".join(_col_names)
_figures_path = get_figures_path() / "comparisons" / _comparison_key
_figures_path.mkdir(parents=True, exist_ok=True)
print(f"\nFigures → {_figures_path}")

In [ ]:
# ── Load linear baseline and augment runs / _col_names ────────────────────────
_LIN_CACHE = Path("/gluster/home/sgoncalves/PinpointDetector/data/weights"
                  "/gravnet_regression_faser_huber1.0_truth2_nofaser_final"
                  "/linear_baseline_cache.npz")
_lin         = np.load(_LIN_CACHE)
_lin_preds   = _lin["linear_preds"]   # (N, 3)
_lin_tgts    = _lin["targets_linear"] # (N, 3)
_lin_y_pred  = _lin["y_pred"]        # (N,)
_lin_y_true  = _lin_tgts[:, 2] / _lin_tgts[:, 0].clip(min=1e-6)

_LIN_COL = "Linear baseline"
_COLORS.insert(0, "#95A5A6")  # grey; linear is first

def _lin_compute_stats(preds, tgts, y_pred, y_true):
    overall, per_bin = {}, {}
    for i, t in enumerate(TARGETS):
        yt  = tgts[:, i]; yp = preds[:, i]
        res = (yp - yt) / yt.clip(min=1e-6)
        ss_res = np.sum((yt - yp)**2); ss_tot = np.sum((yt - yt.mean())**2)
        overall[t] = {
            "r2":             float(1 - ss_res / ss_tot),
            "median_rel_err": float(np.median(np.abs(res))),
            "std_res":        float(np.std(res)),
            "mean_bias":      float(np.mean(res)),
        }
        bins = []
        for emin, emax in ENERGY_BINS_TEV:
            mask = (yt >= emin) & (yt < emax)
            if mask.sum() < 5: continue
            bins.append({
                "emin": emin, "emax": emax,
                "gauss_sigma": float(np.std(res[mask])),
                "std_res":     float(np.std(res[mask])),
                "mean_res":    float(np.mean(res[mask])),
            })
        per_bin[t] = bins
    y_res = (y_pred - y_true) / y_true.clip(min=1e-6)
    inel  = {
        "r2":        float(1 - np.sum((y_pred-y_true)**2) / np.sum((y_true-y_true.mean())**2)),
        "std_res":   float(np.std(y_res)),
        "mean_bias": float(np.mean(y_res)),
    }
    return overall, per_bin, inel

_lin_overall, _lin_perbin, _lin_inel = _lin_compute_stats(
    _lin_preds, _lin_tgts, _lin_y_pred, _lin_y_true)

_FINAL_LABELS["linear_hits_baseline"] = _LIN_COL
runs.insert(0, {
    "weights_dir": "linear_hits_baseline",
    "overall":      _lin_overall,
    "inelasticity": _lin_inel,
    "per_bin":      _lin_perbin,
    "baseline":     None,
})
_col_names.insert(0, _LIN_COL)

print(f"Linear baseline loaded  (N={len(_lin_preds)} events)")
for t in TARGETS:
    s = _lin_overall[t]
    print(f"  {t:<12}  σ={s['std_res']:.3f}  bias={s['mean_bias']:+.3f}  R²={s['r2']:.3f}")

In [ ]:
import json as _json_m
import datetime as _dt_m

GREEN, RED, RESET = "\033[32m", "\033[31m", "\033[0m"

higher_is_better = {"r2": True, "median_rel_err": False, "std_res": False, "mean_bias": False}

def _fmt_delta(v_base, v_cmp, metric):
    diff   = v_cmp - v_base
    if metric == "mean_bias":
        good = abs(v_cmp) < abs(v_base)
    else:
        good = diff > 0 if higher_is_better[metric] else diff < 0
    colour = GREEN if good else RED
    return f"{colour}{diff:>+10.4f}{RESET}"

# Header
col_w = 12
hdr_vals = "".join(f"{c:>{col_w}}" for c in _col_names)
hdr_deltas = "".join(f"  Δ vs {_col_names[1]:>{col_w-7}}" for c in _col_names[2:])
print(f"{'Target':<12} {'Metric':<14}{hdr_vals}{hdr_deltas}")
print("─" * (26 + col_w * len(runs) + 12 * (len(runs) - 1)))

for t in TARGETS:
    for metric, label in [("r2", "R²"), ("median_rel_err", "Med|RelErr|"),
                           ("std_res", "Std(res)"), ("mean_bias", "Bias")]:
        vals   = "".join(f"{r['overall'][t][metric]:>{col_w}.4f}" for r in runs)
        deltas = "".join(f"  {_fmt_delta(runs[1]['overall'][t][metric], r['overall'][t][metric], metric):>10}"
                         for r in runs[2:])
        print(f"{t:<12} {label:<14}{vals}{deltas}")
    print()

print(f"{'Inelasticity y':<12} {'Metric':<14}{hdr_vals}{hdr_deltas}")
print("─" * (26 + col_w * len(runs) + 12 * (len(runs) - 1)))
for metric, label in [("r2", "R²"), ("std_res", "Std(res)"), ("mean_bias", "Bias")]:
    vals   = "".join(f"{r['inelasticity'][metric]:>{col_w}.4f}" for r in runs)
    deltas = "".join(f"  {_fmt_delta(runs[1]['inelasticity'][metric], r['inelasticity'][metric], metric):>10}"
                     for r in runs[2:])
    print(f"{'':12} {label:<14}{vals}{deltas}")

print()
print("── Metric legend ──────────────────────────────────────────────────────────")
print("  R²           Coefficient of determination.  1 = perfect.           ↑ better")
print("  Med|RelErr|  Median |pred − true| / true.   0 = perfect.           ↓ better")
print("  Std(res)     Std dev of (pred − true) / true (resolution).         ↓ better")
print("  Bias         Mean    (pred − true) / true   (systematic offset).   → 0")
print("  Δ vs run 0   Green = improvement over first run in COMPARE list.")
print("──────────────────────────────────────────────────────────────────────────")

# ── Save metrics table to JSON ────────────────────────────────────────────────
_ref = runs[1]  # GravNet baseline is delta reference
_metrics_data = {
    "meta": {
        "col_names":   _col_names,
        "delta_ref":   _col_names[1],
        "timestamp":   _dt_m.datetime.now().isoformat(),
    },
    "per_target":    {},
    "inelasticity_y": {},
}
for _t in TARGETS:
    _metrics_data["per_target"][_t] = {}
    for _metric in ("r2", "median_rel_err", "std_res", "mean_bias"):
        _v_ref = float(_ref["overall"][_t][_metric])
        _metrics_data["per_target"][_t][_metric] = {
            _col_names[_i]: {
                "value": float(_r["overall"][_t][_metric]),
                "delta": float(_r["overall"][_t][_metric]) - _v_ref,
            }
            for _i, _r in enumerate(runs)
        }
for _metric in ("r2", "std_res", "mean_bias"):
    _v_ref = float(_ref["inelasticity"][_metric])
    _metrics_data["inelasticity_y"][_metric] = {
        _col_names[_i]: {
            "value": float(_r["inelasticity"][_metric]),
            "delta": float(_r["inelasticity"][_metric]) - _v_ref,
        }
        for _i, _r in enumerate(runs)
    }
_metrics_file = _figures_path / "metrics_summary.json"
_metrics_file.write_text(_json_m.dumps(_metrics_data, indent=2))
print(f"\nSaved: {_metrics_file}")

In [ ]:
# ── Bar chart: overall metrics per target ─────────────────────────────────────
metrics_to_plot = [
    ("r2",             "R²",           "↑"),
    ("median_rel_err", "Med|RelErr|",   "↓"),
    ("std_res",        "Std(res)",      "↓"),
    ("mean_bias",      "Mean bias",     "→0"),
]

x     = np.arange(len(TARGETS))
width = 0.8 / len(runs)

fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(5 * len(metrics_to_plot), 5))

for ax, (metric, label, indicator) in zip(axes, metrics_to_plot):
    for i, r in enumerate(runs):
        vals = [r["overall"][t][metric] for t in TARGETS]
        offset = (i - len(runs) / 2 + 0.5) * width
        ax.bar(x + offset, vals, width * 0.9,
               label=display_name(r),
               color=_COLORS[i % len(_COLORS)], alpha=_ALPHA)
    ax.set_xticks(x)
    ax.set_xticklabels(TARGET_LATEX)
    ax.set_ylabel(label, fontsize=11)
    ax.set_title(f"{label} ({indicator} better)", fontsize=15)
    if metric == "std_res":
        ax.set_yscale("log")
    elif metric == "mean_bias":
        ax.set_yscale("symlog", linthresh=0.05)
    ax.tick_params(labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(True, axis="y", linestyle=":", alpha=0.3)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=min(len(runs), 3),
           bbox_to_anchor=(0.5, -0.12), frameon=False, fontsize=11)
plt.tight_layout()
plt.savefig(_figures_path / "bar_metrics.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {_figures_path / 'bar_metrics.png'}")

In [ ]:
import torch
import matplotlib.colors as mcolors

def _preds_to_physical(preds_raw, norm_stats):
    preds = torch.from_numpy(preds_raw)
    if norm_stats is not None:
        log_E_nu = preds[:, 0] * norm_stats["sigma_enu"]   + norm_stats["mu_enu"]
        logit_y  = preds[:, 1] * norm_stats["sigma_logit"] + norm_stats["mu_logit"]
    else:
        log_E_nu = preds[:, 0]
        logit_y  = preds[:, 1]
    E_nu  = 10 ** log_E_nu
    y     = torch.sigmoid(logit_y)
    E_roe = y * E_nu          # y = inelasticity = E_roe/E_nu
    E_lep = (1 - y) * E_nu
    return torch.stack([E_nu, E_lep, E_roe], dim=1).numpy(), y.numpy()

weights_path_base = get_weights_path()
run_data = []
for i, r in enumerate(runs):
    if r["baseline"] is None: continue  # linear — added separately below
    ckpt  = torch.load(weights_path_base / r["weights_dir"] / "best_model.pt", map_location="cpu", weights_only=False)
    cache = np.load(weights_path_base / r["weights_dir"] / f"inference_cache_{r['cache_key']}.npz")
    preds_linear, y_pred = _preds_to_physical(cache["preds_raw"], ckpt.get("norm_stats"))
    targets_linear = cache["targets_linear"]
    run_data.insert(0, {
        "col_name":       _col_names[i],
        "preds_linear":   preds_linear,
        "targets_linear": targets_linear,
        "y_pred":         y_pred,
        "y_true":         targets_linear[:, 2] / targets_linear[:, 0].clip(min=1e-6),
    })

# 4 parity plots per run: E_nu, y (inelasticity), E_lep, E_roe
plot_targets = [
    ("E_nu",         r"$E_\nu$",                                   "TeV", True),
    ("inelasticity", r"$y = E_\mathrm{roe}/E_\nu$  (inelasticity)", "",   False),
    ("E_lepton",     r"$E_\mathrm{lep}$",                          "TeV", True),
    ("E_roe",        r"$E_\mathrm{roe}$",                          "TeV", True),
]

cmap_parity = mcolors.LinearSegmentedColormap.from_list(
    'trunc', plt.cm.ocean(np.linspace(0.3, 0.9, 100)))

def _draw_parity_ax(ax, y_true_p, y_pred_p, col_name, key, latex, units, log_scale):
    ss_res    = np.sum((y_true_p - y_pred_p) ** 2)
    ss_tot    = np.sum((y_true_p - y_true_p.mean()) ** 2)
    r2        = 1 - ss_res / ss_tot
    pearson_r = np.corrcoef(y_true_p, y_pred_p)[0, 1]
    ax.set_facecolor("#F5F5F5")
    if log_scale:
        pos_mask = (y_true_p > 0) & (y_pred_p > 0)
        xt, xp = y_true_p[pos_mask], y_pred_p[pos_mask]
        lo = max(xt.min(), 1e-6)
        hi = xt.max()
        hb = ax.hexbin(xt, xp,
                       xscale='log', yscale='log',
                       gridsize=120, cmap=cmap_parity, bins='log',
                       mincnt=1, alpha=0.8,
                       extent=[np.log10(lo), np.log10(hi),
                               np.log10(lo), np.log10(hi)])
        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)
        ax.plot([lo, hi], [lo, hi], "k--", linewidth=0.5)
    else:
        pred_limit = float(np.percentile(y_pred_p, 99))
        hi_i = min(max(np.percentile(y_true_p, 99), pred_limit), 3.0)
        pad  = 0.05
        hb = ax.hexbin(y_true_p, y_pred_p,
                       gridsize=100, cmap=cmap_parity, bins='log',
                       mincnt=1, alpha=0.8,
                       extent=[0, hi_i, 0, hi_i])
        ax.set_xlim(-pad, hi_i + pad)
        ax.set_ylim(-pad, hi_i + pad)
        ax.plot([0, hi_i], [0, hi_i], "k--", linewidth=0.5)
    plt.colorbar(hb, ax=ax, label='Counts')
    ax.set_aspect('equal')
    xlabel = rf"True {latex}" + (f" [{units}]" if units else "")
    ylabel = rf"Predicted {latex}" + (f" [{units}]" if units else "")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(f"{col_name} \u2014 {key.replace('_', ' ')}")
    ax.text(0.05, 0.95, f"$R^2$ = {r2:.4f}\n$r$ = {pearson_r:.4f}",
            transform=ax.transAxes, va="top", fontsize=11,
            bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(True, linestyle=':', linewidth=0.8, color='gray', alpha=0.3)

fig, axes = plt.subplots(len(run_data), len(plot_targets),
                         figsize=(6 * len(plot_targets), 5 * len(run_data)))
if len(run_data) == 1:
    axes = axes.reshape(1, -1)

for row_i, rd in enumerate(run_data):
    tl = rd["targets_linear"]
    pl = rd["preds_linear"]
    for col_i, (key, latex, units, log_scale) in enumerate(plot_targets):
        ax = axes[row_i, col_i]
        if key == "inelasticity":
            y_true_p, y_pred_p = rd["y_true"], rd["y_pred"]
        elif key == "E_nu":
            y_true_p, y_pred_p = tl[:, 0], pl[:, 0]
        elif key == "E_lepton":
            y_true_p, y_pred_p = tl[:, 1], pl[:, 1]
        else:
            y_true_p, y_pred_p = tl[:, 2], pl[:, 2]
        _draw_parity_ax(ax, y_true_p, y_pred_p, rd["col_name"], key, latex, units, log_scale)

plt.tight_layout()
plt.savefig(_figures_path / "parity_comparison.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {_figures_path / 'parity_comparison.png'}")
# ── Append linear baseline to run_data ────────────────────────────────────────
run_data.insert(0, {
    "col_name":       _LIN_COL,
    "preds_linear":   _lin_preds,
    "targets_linear": _lin_tgts,
    "y_pred":         _lin_y_pred,
    "y_true":         _lin_y_true,
})

In [ ]:
# ── Pearson r for all models and targets, with deltas vs GravNet baseline ─────
def _pearson(rd, tidx):
    if tidx == 3:
        return np.corrcoef(rd["y_true"], rd["y_pred"])[0, 1]
    return np.corrcoef(rd["targets_linear"][:, tidx],
                       rd["preds_linear"][:, tidx])[0, 1]

base_rd  = run_data[3]  # GravNet baseline
r_base   = [_pearson(base_rd, i) for i in range(4)]
all_rs   = {rd["col_name"]: [_pearson(rd, i) for i in range(4)] for rd in run_data}

TARGETS_R = ["E_nu", "E_lep", "E_roe", "y"]
hdr = f"{'Model':<25}" + "".join(f"{t:>9}" for t in TARGETS_R)
print(hdr);  print("─" * len(hdr))
for rd in run_data:
    rs = all_rs[rd["col_name"]]
    print(f"{rd['col_name']:<25}" + "".join(f"{r:>9.4f}" for r in rs))

print()
print(f"{'Δ vs GravNet base':<25}")
for rd in run_data:
    rs = all_rs[rd["col_name"]]
    deltas = [rs[i] - r_base[i] for i in range(4)]
    print(f"  {rd['col_name']:<23}" + "".join(f"{d:>+9.4f}" for d in deltas))


In [ ]:
# ── Parity plot: E_lep and E_roe only (for report) ──────────────────────────
plot_targets_2col = [
    ("E_lepton", r"$E_\mathrm{lep}$", "TeV", True),
    ("E_roe",    r"$E_\mathrm{roe}$", "TeV", True),
]

fig2, axes2 = plt.subplots(len(run_data), len(plot_targets_2col),
                            figsize=(6 * len(plot_targets_2col), 5 * len(run_data)))
if len(run_data) == 1:
    axes2 = axes2.reshape(1, -1)

for row_i, rd in enumerate(run_data):
    tl = rd["targets_linear"]
    pl = rd["preds_linear"]
    for col_i, (key, latex, units, log_scale) in enumerate(plot_targets_2col):
        ax = axes2[row_i, col_i]
        y_true_p, y_pred_p = (tl[:, 1], pl[:, 1]) if key == "E_lepton" else (tl[:, 2], pl[:, 2])
        _draw_parity_ax(ax, y_true_p, y_pred_p, rd["col_name"], key, latex, units, log_scale)

plt.tight_layout()
plt.savefig(_figures_path / "parity_Elep_Eroe.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {_figures_path / 'parity_Elep_Eroe.png'}")

In [ ]:
# ── Per-bin resolution (Gaussian σ) and bias (mean residual) ─────────────
def _plot_per_bin(metric_fn, ylabel, suptitle, fname, axhline=False):
    fig, axes = plt.subplots(1, len(TARGETS), figsize=(6 * len(TARGETS), 5))
    for ax, target, latex in zip(axes, TARGETS, TARGET_LATEX):
        for i, r in enumerate(runs):
            bins = r["per_bin"][target]
            xs = [np.sqrt(b["emin"] * b["emax"]) for b in bins]
            ys = [metric_fn(b) for b in bins]
            ax.plot(xs, ys, "o-", color=_COLORS[i % len(_COLORS)], linewidth=1.3, markersize=5,
                    markerfacecolor="white", markeredgewidth=1.2, alpha=0.9,
                    label=display_name(r))
        if axhline:
            ax.axhline(0, color="k", linestyle=":", linewidth=0.8, alpha=0.8)
        ax.set_xscale("log")
        ax.set_xlabel(rf"True {latex} [TeV]  (log scale)", fontsize=11)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.set_title(target, fontsize=15)
        ax.tick_params(labelsize=8)
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(True, linestyle=":", alpha=0.3)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=min(len(runs), 3),
               bbox_to_anchor=(0.5, -0.12), frameon=False, fontsize=11)
    plt.suptitle(suptitle, fontsize=15)
    plt.tight_layout()
    plt.savefig(_figures_path / fname, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {_figures_path / fname}")

_plot_per_bin(
    metric_fn=lambda b: b["gauss_sigma"] if b["gauss_sigma"] is not None else b["std_res"],
    ylabel=r"Resolution $\sigma$ (Gaussian fit)",
    suptitle="Per-bin resolution",
    fname="resolution.png",
)
_plot_per_bin(
    metric_fn=lambda b: b["mean_res"],
    ylabel="Mean (pred - true) / true",
    suptitle="Per-bin bias",
    fname="bias.png",
    axhline=True,
)

In [ ]:
import pandas as pd

def _build_df(rd):
    tl = rd["targets_linear"]
    pl = rd["preds_linear"]
    y_true = rd["y_true"]
    y_pred = rd["y_pred"]
    rows = []
    for i in range(len(tl)):
        rows.append({
            "E_nu":        tl[i, 0],
            "y_true":      y_true[i],
            "abserr_Enu":  abs((pl[i, 0] - tl[i, 0]) / max(tl[i, 0], 1e-6)),
            "abserr_Elep": abs((pl[i, 1] - tl[i, 1]) / max(tl[i, 1], 1e-6)),
            "abserr_Eroe": abs((pl[i, 2] - tl[i, 2]) / max(tl[i, 2], 1e-6)),
            "abserr_y":    abs((y_pred[i] - y_true[i]) / max(y_true[i], 1e-6)),
        })
    df = pd.DataFrame(rows)
    Q = 0.25
    df["worst_Enu"]      = df["abserr_Enu"] >= df["abserr_Enu"].quantile(1 - Q)
    df["worst_y"]        = df["abserr_y"]   >= df["abserr_y"].quantile(1 - Q)
    df["best_Enu"]       = df["abserr_Enu"] <= df["abserr_Enu"].quantile(Q)
    df["best_y"]         = df["abserr_y"]   <= df["abserr_y"].quantile(Q)
    df["worst_Enu_only"] = df["worst_Enu"] & ~df["worst_y"]
    df["worst_y_only"]   = df["worst_y"]   & ~df["worst_Enu"]
    df["worst_both"]     = df["worst_Enu"] & df["worst_y"]
    df["best_both"]      = df["best_Enu"]  & df["best_y"]
    return df

run_dfs = [_build_df(rd) for rd in run_data]

# ── Error correlation: one scatter per run ─────────────────────────────────────
fig, axes = plt.subplots(1, len(run_data), figsize=(6 * len(run_data), 5),
                         sharey=True)
if len(run_data) == 1:
    axes = [axes]

for ax, df, rd in zip(axes, run_dfs, run_data):
    r   = df["abserr_Enu"].corr(df["abserr_y"])
    col = _COLORS[run_data.index(rd) % len(_COLORS)]
    ax.scatter(df["abserr_Enu"], df["abserr_y"],
               s=3, alpha=_ALPHA, color=col, linewidths=0)
    ax.axvline(df["abserr_Enu"].quantile(0.75), color="#e05c5c",
               linestyle="--", linewidth=1.0, label="Worst $E_\\nu$ (75th pct)")
    ax.axhline(df["abserr_y"].quantile(0.75),   color="#e8a838",
               linestyle="--", linewidth=1.0, label=r"Worst $y$ (75th pct)")
    ax.set_xlim(0, np.percentile(df["abserr_Enu"], 99))
    ax.set_ylim(0, np.percentile(df["abserr_y"],   99))
    ax.set_xlabel(r"$|$rel err$|$  $E_\nu$", fontsize=11)
    ax.set_ylabel(r"$|$rel err$|$  $y = E_\mathrm{lep}/E_\nu$", fontsize=11)
    ax.set_title(f"{rd['col_name']}  ($r={r:.3f}$)", fontsize=15)
    ax.tick_params(labelsize=8)
    ax.legend(fontsize=11, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(True, linestyle=":", alpha=0.3)

plt.suptitle(r"Error correlation: $E_\nu$ vs lepton fraction $y = E_\mathrm{lep}/E_\nu$  (= $1 - y_\mathrm{Bjorken}$)", fontsize=15)
plt.tight_layout()
plt.savefig(_figures_path / "error_correlation.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {_figures_path / 'error_correlation.png'}")

In [ ]:
# ── Failure mode breakdown: % events per group, per run ───────────────────────
group_cols   = ["best_both", "worst_Enu_only", "worst_y_only", "worst_both"]
group_labels = ["Best both", "Worst $E_\\nu$ only", "Worst $y$ only", "Worst both"]

pcts = np.array([[100 * df[g].mean() for g in group_cols] for df in run_dfs])  # [n_runs, 4]

x     = np.arange(len(group_labels))
width = 0.8 / len(runs)

fig, ax = plt.subplots(figsize=(9, 5))
for i, (rd, row) in enumerate(zip(run_data, pcts)):
    offset = (i - len(runs) / 2 + 0.5) * width
    bars = ax.bar(x + offset, row, width * 0.9,
                  label=rd["col_name"], color=_COLORS[i % len(_COLORS)], alpha=_ALPHA)
    for bar, v in zip(bars, row):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f"{v:.1f}%", ha="center", va="bottom", fontsize=11)

ax.set_xticks(x)
ax.set_xticklabels(group_labels, fontsize=11)
ax.set_ylabel("% of val events", fontsize=11)
ax.set_title("Failure mode breakdown per run  (25th / 75th percentile thresholds)", fontsize=15)
ax.tick_params(labelsize=8)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(True, axis="y", linestyle=":", alpha=0.3)
ax.legend(frameon=False, fontsize=11)
plt.tight_layout()
plt.savefig(_figures_path / "failure_modes.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {_figures_path / 'failure_modes.png'}")

# Numeric summary
print(f"\n{'Group':<22}" + "".join(f"{rd['col_name']:>16}" for rd in run_data))
for j, lbl in enumerate(group_labels):
    row_str = f"{lbl.replace('$',''):<22}" + "".join(f"{pcts[i, j]:>15.1f}%" for i in range(len(runs)))
    print(row_str)

In [ ]:
# ── GravNet vs linear baseline ────────────────────────────────────────────────
print(f"{'Run':<40} {'Target':<12} {'GravNet σ':>10} {'Linear σ':>10} {'Improvement':>12}")
print("-" * 86)
for r in runs:
    if r["baseline"] is None: continue  # linear model has no GravNet comparison
    name = display_name(r)
    for t in TARGETS:
        b = r["baseline"][t]
        imp = (b["linear_std_res"] - b["gravnet_std_res"]) / b["linear_std_res"] * 100
        print(f"{name:<40} {t:<12} {b['gravnet_std_res']:>10.3f} {b['linear_std_res']:>10.3f} {imp:>+11.1f}%")
    print()

In [ ]:
# ── Fraction of headroom recovered (requires COMPARE = [Exp1, Exp4, Exp5]) ───
# Headroom = gap between baseline (Exp1) and truth-label ceiling (Exp4).
# Fraction recovered = (σ_Exp1 − σ_Exp5) / (σ_Exp1 − σ_Exp4)
# Bootstrap CI uses paired resampling across all three models simultaneously.

if len(run_data) < 4:
    print("Headroom cell requires exactly 3 runs in COMPARE: [Exp1, Exp4, Exp5]. Skipping.")
else:
    rng = np.random.default_rng(42)
    print(f"Fraction of headroom recovered  (Exp1→Exp4 gap, Exp5 relative to gap)")
    print(f"{'Target':<10}  {'Fraction':>10}  {'95% CI':^20}  {'σ_Exp1':>8}  {'σ_Exp4':>8}  {'σ_Exp5':>8}")
    print("─" * 76)
    for target_idx, name in zip([1, 2], ["E_lep", "E_roe"]):
        yt  = run_data[1]["targets_linear"][:, target_idx]  # Exp1
        yp1 = run_data[1]["preds_linear"][:, target_idx]    # Exp1 baseline
        yp4 = run_data[2]["preds_linear"][:, target_idx]    # Exp4 truth ceiling
        yp5 = run_data[3]["preds_linear"][:, target_idx]    # Exp5 pipeline

        n = len(yt)
        fracs = []
        for _ in range(1000):
            idx = rng.integers(0, n, n)
            s1 = np.std((yp1[idx] - yt[idx]) / yt[idx])
            s4 = np.std((yp4[idx] - yt[idx]) / yt[idx])
            s5 = np.std((yp5[idx] - yt[idx]) / yt[idx])
            gap = s1 - s4
            fracs.append((s1 - s5) / gap if abs(gap) > 1e-9 else np.nan)

        fracs = np.array(fracs)
        fracs = fracs[np.isfinite(fracs)]
        med_f    = float(np.median(fracs))
        lo_f, hi_f = np.percentile(fracs, [2.5, 97.5])

        s1_pt = np.std((yp1 - yt) / yt)
        s4_pt = np.std((yp4 - yt) / yt)
        s5_pt = np.std((yp5 - yt) / yt)
        print(f"{name:<10}  {med_f:>9.1%}  [{lo_f:.1%}, {hi_f:.1%}]"
              f"  {s1_pt:>8.3f}  {s4_pt:>8.3f}  {s5_pt:>8.3f}")

In [ ]:
# ── Resolution and bias vs inelasticity y ─────────────────────────────────────────
# err_fn signature: (residuals_in_bin, n_bin) -> scalar
# Resolution uses σ/√(2n) (std-error of the std estimator).
# Bias uses σ/√n (SEM). These are NOT interchangeable.
def _plot_vs_y(stat_fn, err_fn, ylabel, fname, legend_fontsize=15, axhline=False):
    y_bins    = np.linspace(0, 1, 11)
    y_centers = 0.5 * (y_bins[:-1] + y_bins[1:])
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.7))
    for ax, target_idx, latex in zip(
            axes, [1, 2],
            [r"$E_\mathrm{lep}$", r"$E_\mathrm{roe}$"]):
        for i, rd in enumerate(run_data):
            yt  = rd["targets_linear"][:, target_idx]
            yp  = rd["preds_linear"][:, target_idx]
            res = (yp - yt) / yt
            y_t = rd["y_true"]
            vals, errs = [], []
            for ymin, ymax in zip(y_bins[:-1], y_bins[1:]):
                mask  = (y_t >= ymin) & (y_t < ymax)
                n_bin = int(mask.sum())
                if n_bin < 20:
                    vals.append(np.nan); errs.append(np.nan); continue
                vals.append(float(stat_fn(res[mask])))
                errs.append(float(err_fn(res[mask], n_bin)))
            col = _COLORS[i % len(_COLORS)]
            ax.errorbar(y_centers, vals, yerr=errs,
                        fmt='o-', label=rd["col_name"], color=col,
                        alpha=0.9, capsize=3, markersize=4, linewidth=1.8,
                        markerfacecolor=col, markeredgecolor=col,
                        markeredgewidth=0.8, zorder=3)
        if axhline:
            ax.axhline(0, color="k", linestyle=":", linewidth=0.8, alpha=0.8)
        ax.set_xlim(-0.02, 1.02)
        ax.set_xlabel(r"True inelasticity $y$", fontsize=11)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.set_title(latex, fontsize=legend_fontsize, pad=4)
        ax.tick_params(labelsize=8)
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(True, linestyle=":", alpha=0.3)
    axes[0].legend(frameon=False, fontsize=legend_fontsize, loc="upper left")
    axes[1].legend(frameon=False, fontsize=legend_fontsize, loc="upper right")
    plt.tight_layout(pad=0.8, w_pad=1.5)
    plt.savefig(_figures_path / fname, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {_figures_path / fname}")

_plot_vs_y(
    stat_fn=np.std,
    err_fn=lambda r, n: np.std(r) / np.sqrt(2 * n),
    ylabel=r"$\sigma(E)/E$",
    fname="resolution_vs_y.png",
    legend_fontsize=15,
)
_plot_vs_y(
    stat_fn=np.mean,
    err_fn=lambda r, n: np.std(r) / np.sqrt(n),
    ylabel=r"Mean $(E_\mathrm{pred} - E_\mathrm{true})\,/\,E_\mathrm{true}$",
    fname="bias_vs_y.png",
    legend_fontsize=14,
    axhline=True,
)

In [ ]:
# ── Scatter residual vs true quantity: per-run panels ──────────────────────
# target_getter(rd) -> (yt, yp, x_vals)
# res_fn(yt, yp)    -> residuals array
# clip_rd: if None, clip = max 99th pct across runs_subset; else clip from that rd alone
def _plot_residual_panels(
    runs_subset, target_getter, res_fn, bins, logx,
    xlabel, ylabel, fname, clip_pct=99, clip_rd=None,
):
    bin_centers = (np.sqrt(bins[:-1] * bins[1:]) if logx
                   else 0.5 * (bins[:-1] + bins[1:]))
    clip_source = [clip_rd] if clip_rd is not None else runs_subset
    clip = max(
        float(np.percentile(np.abs(res_fn(*target_getter(rd)[:2])), clip_pct))
        for rd in clip_source
    )
    fig, axes = plt.subplots(1, len(runs_subset),
                             figsize=(5 * len(runs_subset), 5), sharey=True)
    if len(runs_subset) == 1:
        axes = [axes]
    for ax, rd in zip(axes, runs_subset):
        yt, yp, x_vals = target_getter(rd)
        res = res_fn(yt, yp)
        ax.scatter(x_vals, res, s=2, alpha=0.4, color='steelblue', linewidths=0, rasterized=True)
        means = [np.mean(res[(x_vals >= lo) & (x_vals < hi)])
                 if ((x_vals >= lo) & (x_vals < hi)).sum() >= 20 else np.nan
                 for lo, hi in zip(bins[:-1], bins[1:])]
        ax.plot(bin_centers, means, 'o-', color="k", linewidth=1.8,
                markersize=4, zorder=3, label='Bin mean')
        ax.axhline(0, color="k", linestyle="--", linewidth=0.8, alpha=0.5)
        if logx:
            ax.set_xscale('log')
            ax.set_xlim(bins[0], bins[-1])
        else:
            ax.set_xlim(bins[0] - 0.02, bins[-1] + 0.02)
        ax.set_ylim(-clip, clip)
        ax.set_xlabel(xlabel, fontsize=11)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.set_title(rd["col_name"], fontsize=13)
        ax.legend(frameon=False, fontsize=10)
        ax.spines[['top', 'right']].set_visible(False)
        ax.grid(True, linestyle=':', linewidth=0.8, color='gray', alpha=0.3)
    plt.tight_layout()
    plt.savefig(_figures_path / fname, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {_figures_path / fname}")

# Inelasticity y residual (absolute) vs true y
_plot_residual_panels(
    runs_subset=[run_data[3], run_data[1], run_data[2]],
    target_getter=lambda rd: (rd["y_true"], rd["y_pred"], rd["y_true"]),
    res_fn=lambda yt, yp: yp - yt,
    bins=np.linspace(0, 1, 11),
    logx=False,
    xlabel=r"True $y$",
    ylabel=r"$y_\mathrm{pred} - y_\mathrm{true}$",
    fname="y_residual_vs_ytrue_base_truth.png",
    clip_pct=99,
)
# E_nu relative residual vs true y
_plot_residual_panels(
    runs_subset=[run_data[3], run_data[2]],
    target_getter=lambda rd: (rd["targets_linear"][:, 0], rd["preds_linear"][:, 0], rd["y_true"]),
    res_fn=lambda yt, yp: (yp - yt) / yt,
    bins=np.linspace(0, 1, 11),
    logx=False,
    xlabel=r"True $y$",
    ylabel=r"$(E_{\nu,\mathrm{pred}} - E_{\nu,\mathrm{true}})\,/\,E_{\nu,\mathrm{true}}$",
    fname="Enu_residual_vs_ytrue_base_truth.png",
    clip_pct=99,
)
# E_lep relative residual vs true E_lep (clip anchored to baseline run)
_plot_residual_panels(
    runs_subset=[run_data[3], run_data[1], run_data[2]],
    target_getter=lambda rd: (rd["targets_linear"][:, 1], rd["preds_linear"][:, 1], rd["targets_linear"][:, 1]),
    res_fn=lambda yt, yp: (yp - yt) / yt,
    bins=np.logspace(np.log10(0.1), np.log10(2.0), 11),
    logx=True,
    xlabel=r"True $E_\mathrm{lep}$ [TeV]",
    ylabel=r"$(E_{\mathrm{lep,pred}} - E_{\mathrm{lep,true}})\,/\,E_{\mathrm{lep,true}}$",
    fname="Elep_residual_vs_Elep_base_truth_pipeline.png",
    clip_pct=90,
    clip_rd=run_data[3],
)
# E_roe relative residual vs true E_roe (clip anchored to baseline run)
_plot_residual_panels(
    runs_subset=[run_data[3], run_data[1], run_data[2]],
    target_getter=lambda rd: (rd["targets_linear"][:, 2], rd["preds_linear"][:, 2], rd["targets_linear"][:, 2]),
    res_fn=lambda yt, yp: (yp - yt) / yt,
    bins=np.logspace(np.log10(0.01), np.log10(2.0), 11),
    logx=True,
    xlabel=r"True $E_\mathrm{roe}$ [TeV]",
    ylabel=r"$(E_{\mathrm{roe,pred}} - E_{\mathrm{roe,true}})\,/\,E_{\mathrm{roe,true}}$",
    fname="Eroe_residual_vs_Eroe_base_truth_pipeline.png",
    clip_pct=90,
    clip_rd=run_data[3],
)

In [ ]:
targets_info = [
    (0, r"True $E_\nu$ [TeV]",          r"$\delta E_\nu$",          0.05, 2.0, "Enu_residual_vs_Enu_overlay.png"),
    (1, r"True $E_\mathrm{lep}$ [TeV]", r"$\delta E_\mathrm{lep}$", 0.1,  2.0, "Elep_residual_vs_Elep_overlay.png"),
    (2, r"True $E_\mathrm{roe}$ [TeV]", r"$\delta E_\mathrm{roe}$", 0.01, 2.0, "Eroe_residual_vs_Eroe_overlay.png"),
]
# fields: col_idx, xlabel, ylabel, clipx1, clipx2, fname
# y-axis: delta_E = (E_pred - E_true) / E_true  — define in caption

# Colours: grey=linear baseline, blue=base GravNet, red=+classifier, black=+truth labels
_PLT_COLORS     = ["#95A5A6", "#4477AA", "#EE6677", "#353D4C"]
_PLT_LINESTYLES = ["--", "-", "-.", ":"]

def _plot_panel(ax, col_idx, xlabel, ylabel, clipx1, clipx2):
    e_bins    = np.logspace(np.log10(clipx1), np.log10(clipx2), 11)
    e_centers = np.sqrt(e_bins[:-1] * e_bins[1:])

    all_means, handles = [], []
    for i, (rd, label) in enumerate(zip(_runs_subset, labels)):
        yt  = rd["targets_linear"][:, col_idx]
        yp  = rd["preds_linear"][:, col_idx]
        res = (yp - yt) / yt
        means, errs = [], []
        for lo, hi in zip(e_bins[:-1], e_bins[1:]):
            m = res[(yt >= lo) & (yt < hi)]
            if len(m) >= 20:
                means.append(m.mean())
                errs.append(m.std() / np.sqrt(len(m)))
            else:
                means.append(np.nan); errs.append(np.nan)
        if i > 0:
            all_means.extend([v for v in means if not np.isnan(v)])
        h = ax.errorbar(e_centers, means, yerr=errs,
                        fmt='.', color=_PLT_COLORS[i % len(_PLT_COLORS)],
                        linestyle=_PLT_LINESTYLES[i % len(_PLT_LINESTYLES)],
                        linewidth=1.5, markersize=8, capsize=3, label=label, zorder=3)
        handles.append(h)

    pad = 0.2
    ylo = min(all_means) - pad * (max(all_means) - min(all_means))
    yhi = max(all_means) + pad * (max(all_means) - min(all_means))
    ylo = min(ylo, -0.02)
    yhi = max(yhi,  0.02)

    ax.axhline(0, color='k', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.set_xscale('log')
    ax.set_xlim(clipx1, clipx2)
    ax.set_ylim(ylo, yhi)
    ax.set_xlabel(xlabel, fontsize=18)
    ax.set_ylabel(ylabel, fontsize=22)
    ax.tick_params(axis='both', labelsize=14)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(True, linestyle=':', linewidth=0.8, color='gray', alpha=0.3)
    return handles

_runs_subset = [run_data[0], run_data[3], run_data[1], run_data[2]]  # linear, base, +classifier, +truth labels
labels       = ['linear', 'base', '+classifier', '+truth labels']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (col_idx, xlabel, ylabel, clipx1, clipx2, fname) in zip(axes, targets_info):
    handles = _plot_panel(ax, col_idx, xlabel, ylabel, clipx1, clipx2)
fig.legend(handles, labels, loc='lower center', ncol=len(_runs_subset),
           frameon=False, fontsize=16, bbox_to_anchor=(0.5, -0.08))
plt.tight_layout()

save_path = _figures_path / "residual_vs_E_overlay_horizontal.png"
plt.savefig(save_path, dpi=350, bbox_inches='tight')
print(f"Saved to {save_path.resolve()}")
plt.show()


In [ ]:
# ── Feature-error correlation comparison (from pre-computed JSON) ─────────────
import json as _fec_json
import numpy as _fec_np
from pathlib import Path as _fec_Path

_fec_file = _fec_Path("feature_error_correlation.json")
_fec_data = _fec_json.loads(_fec_file.read_text())

_feat_labels_fec = [r"$E_\nu$ true", r"$y$ true", r"$\log E_\nu$", "N nodes"]
_row_keys_fec    = ["E_v true", "y true", "log E_v", "N nodes"]
_err_labels_fec  = [r"|err| $E_\nu$",   r"|err| $E_\mathrm{lep}$",
                    r"|err| $E_\mathrm{roe}$", r"|err| $y$",
                    r"res $E_\nu$",     r"res $E_\mathrm{lep}$",
                    r"res $E_\mathrm{roe}$",   r"res $y$"]

_model_order  = ["1_baseline", "4_truth_labels", "5_pipeline"]
_model_titles = {
    "1_baseline":     "GravNet",
    "4_truth_labels": "GravNet (truth labels)",
    "5_pipeline":     "GravNet + classifier",
}

_nf, _ne   = len(_feat_labels_fec), len(_err_labels_fec)
_vmin, _vmax = -0.6, 0.6

_matrices_fec = {
    _mk: _fec_np.array([_fec_data["models"][_mk]["matrix"][r] for r in _row_keys_fec])
    for _mk in _model_order
}

def _draw_heatmap(ax, arr, title, annot_fmt="{:.2f}", thresh=0.35):
    im = ax.imshow(arr, cmap="RdBu_r", vmin=_vmin, vmax=_vmax, aspect="auto")
    ax.set_yticks(range(_nf))
    ax.set_yticklabels(_feat_labels_fec, fontsize=15)
    ax.set_title(title, fontsize=15, pad=3)
    for i in range(_nf):
        for j in range(_ne):
            v = arr[i, j]
            ax.text(j, i, annot_fmt.format(v), ha="center", va="center",
                    fontsize=15, color="white" if abs(v) > thresh else "black")
    ax.spines[["top", "right", "left", "bottom"]].set_visible(False)
    return im

def _add_xticklabels(ax):
    ax.set_xticks(range(_ne))
    ax.set_xticklabels(_err_labels_fec, rotation=35, ha="right", fontsize=15)

def _hcbar(fig, im, label):
    cbar_ax = fig.add_axes([0.2, 0.02, 0.6, 0.022])
    cb = fig.colorbar(im, cax=cbar_ax, orientation="horizontal")
    cb.set_label(label, fontsize=15)

# ── Figure 1: all three models, vertical stack ────────────────────────────────
_fig1, _ax1 = plt.subplots(len(_model_order), 1,
                            figsize=(11, 2.8 * len(_model_order)), sharex=True)
for _ax, _mk in zip(_ax1, _model_order):
    _im1 = _draw_heatmap(_ax, _matrices_fec[_mk], _model_titles[_mk])
_add_xticklabels(_ax1[-1])
plt.tight_layout(rect=[0, 0.07, 1, 1])
_hcbar(_fig1, _im1, "Pearson $r$")
plt.savefig(_figures_path / "feature_error_correlation_comparison.png",
            dpi=350, bbox_inches="tight")
plt.show()
print("Saved: feature_error_correlation_comparison.png")

# ── Figure 2: GravNet baseline + diffs ───────────────────────────────────────
_base_arr  = _matrices_fec["1_baseline"]
_diff_keys = ["4_truth_labels", "5_pipeline"]
_diff_ttls = {
    "4_truth_labels": r"$\Delta$: GravNet (truth labels) $-$ GravNet",
    "5_pipeline":     r"$\Delta$: GravNet + classifier $-$ GravNet",
}

_fig2, _ax2 = plt.subplots(1 + len(_diff_keys), 1,
                            figsize=(11, 2.8 * (1 + len(_diff_keys))), sharex=True)
_im2 = _draw_heatmap(_ax2[0], _base_arr, "GravNet (reference)")
for _ax, _mk in zip(_ax2[1:], _diff_keys):
    _diff = _matrices_fec[_mk] - _base_arr
    _draw_heatmap(_ax, _diff, _diff_ttls[_mk], annot_fmt="{:+.2f}")
_add_xticklabels(_ax2[-1])
plt.tight_layout(rect=[0, 0.07, 1, 1])
_hcbar(_fig2, _im2, r"Pearson $r$  /  $\Delta$ Pearson $r$")
plt.savefig(_figures_path / "feature_error_correlation_diff.png",
            dpi=350, bbox_inches="tight")
plt.show()
print("Saved: feature_error_correlation_diff.png")

# ── Pairwise matrix similarity ────────────────────────────────────────────────
from itertools import combinations as _comb
print("\nPairwise Pearson r between flattened correlation matrices:")
for _i, _j in _comb(_model_order, 2):
    _r = float(_fec_np.corrcoef(_matrices_fec[_i].ravel(), _matrices_fec[_j].ravel())[0, 1])
    print(f"  {_model_titles[_i]:30s} vs {_model_titles[_j]:30s}  r = {_r:.4f}")